In [ ]:
# =============================================================================
# 1. Importación de Librerías & Cliente
# =============================================================================
import pandas as pd
from datetime import datetime, timedelta
import pytz
from google.cloud import bigquery
from google.cloud import storage
from google.api_core.exceptions import NotFound

clientBQ = bigquery.Client()
storage_client = storage.Client()

In [ ]:
#@title Variable Dataset & Tables { run: "auto", display-mode: "form" }
var_project_storage = "prd-izipay-data-storage-pv" #@param {type:"string"}
var_project_sensitive = "prd-izipay-data-sensitive" #@param {type:"string"}
var_project_operation = "prd-izipay-data-operation" #@param {type:"string"}
var_dataset_bi_riesgo = "bi_riesgo" #@param {type:"string"}
var_dataset_master_pii = "master_pii" #@param {type:"string"}
var_dataset_secure_secrets = "secure_secrets" #@param {type:"string"}
var_table_dv_consolidacion_contracargo = "dv_consolidacion_contracargo" #@param {type:"string"}
var_table_iden_party_data_control = "iden_party_data_control" #@param {type:"string"}
var_table_config_protected_data = "config_protected_data" #@param {type:"string"}


In [ ]:
# =============================================================================
# 2. Configuración de fechas de inicio y rango
# =============================================================================
lima_tz = pytz.timezone("America/Lima")
hoy = datetime.now(lima_tz).date()
var_anho = hoy.year
var_mes  = hoy.month

# day_of_week en Python: 0=Lunes, 1=Martes, ..., 6=Domingo
dia_semana = hoy.isoweekday()

# --- fecha_inicio ---
if dia_semana in (1, 2):   # Lunes o Martes
    fecha_inicio = hoy - timedelta(days=3) ## cambiar por 3
else:
    fecha_inicio = hoy - timedelta(days=1) ## cambiar por 1

# --- fecha_fin ---
if dia_semana == 1:         # Lunes
    fecha_fin = hoy - timedelta(days=3)
elif dia_semana == 2:       # Martes
    fecha_fin = hoy
else:                       # Otro día
    fecha_fin = hoy - timedelta(days=1)

print(f"Hoy         : {hoy} (día semana: {dia_semana})")
print(f"Año         : {var_anho}")
print(f"Mes         : {var_mes}")
print(f"fecha_inicio: {fecha_inicio}")
print(f"fecha_fin   : {fecha_fin}")

Hoy         : 2026-04-06 (día semana: 1)
Año         : 2026
Mes         : 4
fecha_inicio: 2026-04-03
fecha_fin   : 2026-04-03


In [ ]:
# =============================================================================
# 3. Configuración de Variables Globales y Diccionario de Reportes
# =============================================================================
var_anho_str = hoy.strftime('%Y')
var_mes_str = hoy.strftime('%m')
var_fecha_file = hoy.strftime('%Y%m%d')
bucket_name = "adls-reportes"
ruta_base = f"Operaciones/Consolidacion_Contracargos/{var_anho_str}/{var_mes_str}/"

print(f"--- Año del proceso: {var_anho_str} ---")
print(f"--- Mes del proceso: {var_mes_str} ---")
print(f"--- Fecha para nombres de archivos: {var_fecha_file} ---")

# Diccionario con las consultas adaptadas para los diferentes reportes
reportes = {
    "reporte_cc_primera_linea": f"""
        with base_contracargo as (
          select
            a.cod_comercio, a.bin, a.cuarteto, a.nom_comercio, a.moneda_trx, a.mto_trx, a.fecha_trx,
            a.cod_autorizacion, a.motivo_cc, a.voucher, a.referencia, a.fecha_devolucion, a.razon_social,
            a.nro_control_contracargo, a.moneda_contracargo, a.mto_contracargo, a.arn_ard, a.fecha_contracargo,
            a.cod_respuesta_ecommerce, a.ind_unicidad_contracargo, a.orden_cc, a.cod_terminal,
            a.metodo_ingreso, a.correo_contacto, a.linea_atencion, a.flag_retencion, a.flag_devolucion
          from `{var_project_storage}.{var_dataset_bi_riesgo}.{var_table_dv_consolidacion_contracargo}` a
          left join `{var_project_sensitive}.{var_dataset_master_pii}.{var_table_iden_party_data_control}` b
               on (a.party_id_izi = b.party_id_izi)
          where a.linea_atencion = 'L1'
            and b.document_number <> '20523621212'
            and a.fecha_contracargo >= '{fecha_inicio}'
            and a.fecha_contracargo <= '{fecha_fin}'
        )
        select
          current_date("America/Lima")                                             as process_date,
          bc.cod_comercio                                                          as cod_comercio,
          concat(bc.bin,'******',bc.cuarteto)                                    as tarjeta_enmascarada,
          bc.nom_comercio                                                          as nom_comercio,
          bc.moneda_trx                                                            as moneda_trx,
          bc.mto_trx                                                               as mto_trx,
          bc.fecha_trx                                                             as fecha_trx,
          bc.cod_autorizacion                                                      as cod_autorizacion,
          bc.motivo_cc                                                             as motivo_cc,
          bc.voucher                                                               as voucher,
          bc.referencia                                                            as referencia,
          bc.fecha_devolucion                                                      as fecha_devolucion,
          nullif(upper(trim(AEAD.DECRYPT_STRING(sec_name.key, bc.razon_social, sec_name.constant))), '') as razon_social,
          bc.nro_control_contracargo                                               as nro_control_contracargo,
          bc.moneda_contracargo                                                    as moneda_contracargo,
          bc.mto_contracargo                                                       as mto_contracargo,
          bc.arn_ard                                                               as arn_ard,
          bc.fecha_contracargo                                                     as fecha_contracargo,
          bc.cod_respuesta_ecommerce                                               as cod_respuesta_ecommerce,
          bc.ind_unicidad_contracargo                                              as ind_unicidad_contracargo,
          bc.orden_cc                                                              as orden_cc,
          bc.cod_terminal                                                          as cod_terminal,
          bc.metodo_ingreso                                                        as metodo_ingreso,
          nullif(upper(trim(AEAD.DECRYPT_STRING(sec.key, bc.correo_contacto, sec.constant))),'') as correo_contacto,
          bc.linea_atencion                                                        as linea_atencion,
          bc.flag_retencion                                                        as flag_retencion,
          bc.flag_devolucion                                                       as flag_devolucion,
          'dv_consolidacion_contracargo - CONTRACARGOS VISA & MASTERCARD'          as record_source,
          current_datetime("America/Lima")                                         as load_date,
          session_user()                                                           as creation_user
        from base_contracargo bc
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec_name  on sec_name.code  = 'C_FULL_NAME'
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec on sec.code = 'C_EMAIL';
    """,

    "reporte_cc_tercera_linea": f"""
        with base_contracargo as (
          select a.*
          from `{var_project_storage}.{var_dataset_bi_riesgo}.{var_table_dv_consolidacion_contracargo}` a
          left join `{var_project_sensitive}.{var_dataset_master_pii}.{var_table_iden_party_data_control}` b
               on (a.party_id_izi = b.party_id_izi)
          where a.linea_atencion = 'L3'
            and b.document_number <> '20523621212'
            and a.fecha_contracargo >= '{fecha_inicio}'
            and a.fecha_contracargo <= '{fecha_fin}'
        )
        select
          current_date("America/Lima")                                             as process_date,
          bc.cod_comercio                                                          as cod_comercio,
          concat(bc.bin,'******',bc.cuarteto)                                    as tarjeta_enmascarada,
          bc.nom_comercio                                                          as nom_comercio,
          bc.moneda_trx                                                            as moneda_trx,
          bc.mto_trx                                                               as mto_trx,
          bc.fecha_trx                                                             as fecha_trx,
          bc.cod_autorizacion                                                      as cod_autorizacion,
          bc.motivo_cc                                                             as motivo_cc,
          bc.voucher                                                               as voucher,
          bc.referencia                                                            as referencia,
          bc.fecha_devolucion                                                      as fecha_devolucion,
          nullif(upper(trim(AEAD.DECRYPT_STRING(sec_name.key, bc.razon_social, sec_name.constant))), '') as razon_social,
          bc.nro_control_contracargo                                               as nro_control_contracargo,
          bc.moneda_contracargo                                                    as moneda_contracargo,
          bc.mto_contracargo                                                       as mto_contracargo,
          bc.arn_ard                                                               as arn_ard,
          bc.fecha_contracargo                                                     as fecha_contracargo,
          bc.cod_respuesta_ecommerce                                               as cod_respuesta_ecommerce,
          bc.ind_unicidad_contracargo                                              as ind_unicidad_contracargo,
          bc.orden_cc                                                              as orden_cc,
          bc.cod_terminal                                                          as cod_terminal,
          bc.metodo_ingreso                                                        as metodo_ingreso,
          nullif(upper(trim(AEAD.DECRYPT_STRING(sec.key, bc.correo_contacto, sec.constant))),'') as correo_contacto,
          bc.linea_atencion                                                        as linea_atencion,
          bc.flag_retencion                                                        as flag_retencion,
          bc.flag_devolucion                                                       as flag_devolucion,
          'dv_consolidacion_contracargo - CONTRACARGOS VISA & MASTERCARD'          as record_source,
          current_datetime("America/Lima")                                         as load_date,
          session_user()                                                           as creation_user
        from base_contracargo bc
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec_name  on sec_name.code  = 'C_FULL_NAME'
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec on sec.code = 'C_EMAIL';
    """,

    "reporte_cc_crm": f"""
        with base as (
            select
                a.cod_comercio, a.ind_unicidad_contracargo, a.nro_control_contracargo, a.arn_ard, a.orden_cc,
                a.marca_tarjeta, a.cod_motivo_contracargo, a.fecha_contracargo, a.mto_contracargo, a.moneda_contracargo,
                a.mto_trx, a.moneda_trx, a.nom_comercio, a.bin, a.cuarteto, a.fecha_trx, a.fecha_proceso,
                a.fecha_retencion, a.fecha_devolucion, a.cod_ica_emisor, a.metodo_ingreso, a.cod_autorizacion,
                a.bancor_emisor, a.motivo_cc, a.categoria_motivo_contracargo, a.voucher, a.cod_respuesta_ecommerce,
                a.juridisccion_trx, a.cod_funcion, a.cod_terminal, a.referencia, a.linea_atencion, a.flag_correo,
                a.atencion_tica, a.atencion_importe, a.flag_atencion_gestion_datos, a.correo_contacto,
                b.document_number, c.key as enc_key, c.constant as enc_constant
            from `{var_project_storage}.{var_dataset_bi_riesgo}.{var_table_dv_consolidacion_contracargo}` a
            left join `{var_project_sensitive}.{var_dataset_master_pii}.{var_table_iden_party_data_control}` b
                 on (a.party_id_izi = b.party_id_izi)
            left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` c
                 on (c.code = 'C_EMAIL')
            where a.fecha_contracargo >= '{fecha_inicio}'
              and a.fecha_contracargo <= '{fecha_fin}'
        )
        select
            current_date("America/Lima")                                                                       as process_date,
            concat('00', cod_comercio)                                                                         as Codigo_de_Comercio,
            'GESTIÓN DE CONTRACARGOS'                                                                          as Proceso,
            'CONTRACARGO ADQUIRENTE'                                                                           as Asunto,
            case when ind_unicidad_contracargo = 'CC UNICO' then nro_control_contracargo
                 else concat(nro_control_contracargo, ' - ', left(ind_unicidad_contracargo, 4), ' - ', orden_cc, ' - ',
                             cast(row_number() over (partition by arn_ard order by arn_ard desc) as string))
            end                                                                                                as Titulo_del_Caso,
            'PRIMER NIVEL'                                                                                     as Nivel_de_Atencion,
            'CARGA MASIVA'                                                                                     as Origin,
            'SOLICITUD'                                                                                        as Tipo_de_Caso,
            'MEDIUM'                                                                                           as Prioridad,
            'CONTRACARGO'                                                                                      as Descripcion,
            marca_tarjeta                                                                                      as Marca_de_Tarjeta,
            case when marca_tarjeta = 'MASTERCARD' then cod_motivo_contracargo
                 else left(cod_motivo_contracargo, 2)
            end                                                                                                as Categoria_de_Controversia,
            nro_control_contracargo                                                                            as Rol_Case_Control,
            'NO'                                                                                               as Completar_Automatico,
            fecha_contracargo                                                                                  as Fecha_de_Controversia,
            mto_contracargo                                                                                    as Importe_CC,
            moneda_contracargo                                                                                 as Moneda_CC,
            mto_trx                                                                                            as Importe_Trx,
            moneda_trx                                                                                         as Moneda_Trx,
            case when left(nom_comercio, 13) = 'ECM*AMZ VIDEO' then '************'
                 else concat(bin, '******', cuarteto)
            end                                                                                                as Tarjeta,
            case when left(nom_comercio, 13) = 'ECM*AMZ VIDEO' then '************'
                 else concat(bin, '******', cuarteto)
            end                                                                                                as Tarjeta_Completa,
            fecha_trx                                                                                          as Fecha_Trx_Venta,
            fecha_proceso                                                                                      as Fecha_de_Proceso_Outgoing,
            fecha_retencion                                                                                    as Fecha_de_Retencion,
            fecha_devolucion                                                                                   as Fecha_de_Devolucion,
            case when marca_tarjeta = 'MASTERCARD' then cod_ica_emisor else bin end                          as Ica_Banco_Emisor_MC,
            metodo_ingreso                                                                                     as Tipo_Trx,
            cod_autorizacion                                                                                   as Autoriza,
            bancor_emisor                                                                                      as Banco_Emisor,
            motivo_cc                                                                                          as Motivo_de_la_Controversia,
            categoria_motivo_contracargo                                                                       as Categoria_General,
            voucher                                                                                            as Voucher,
            cod_respuesta_ecommerce                                                                            as Ecom,
            'EN ANÁLISIS'                                                                                      as Status_Contracargos,
            'EN ANÁLISIS'                                                                                      as Motivo_Resultado_Final,
            concat('CC_ADQ', ' - ', juridisccion_trx, ' - ', cod_funcion)                                      as Jurisdiccion_Solo_Visa,
            case when ind_unicidad_contracargo = 'CC UNICO' then arn_ard
                 else concat(arn_ard, ' - ', left(ind_unicidad_contracargo, 4), ' - ', orden_cc, ' - ',
                             cast(row_number() over (partition by arn_ard order by arn_ard desc) as string))
            end                                                                                                as ARN_ARD,
            cod_terminal                                                                                       as Terminal,
            referencia                                                                                         as Referencia,
            document_number                                                                                    as RUC,
            cod_funcion                                                                                        as Fun,
            linea_atencion                                                                                     as Tipo_Linea,
            flag_correo                                                                                        as Flag_Correo,
            atencion_tica                                                                                      as Atencion_TICA,
            atencion_importe                                                                                   as Atencion_Importe,
            flag_atencion_gestion_datos                                                                        as Atencion_GD,
            orden_cc                                                                                           as Orden,
            nullif(upper(trim(AEAD.DECRYPT_STRING(enc_key, correo_contacto,enc_constant))),'')                 as Correo_Tecnico_del_Comercio,
            'CONTRACARGOS VISA & MASTERCARD'                                                                   as record_source,
            current_datetime("America/Lima")                                                                   as load_date,
            session_user()                                                                                     as creation_user
        from base;
    """,

    "reporte_cc_cartas": f"""
        with base as (
            select
                a.cod_comercio, a.nom_comercio, a.bin, a.cuarteto, a.moneda_trx, a.mto_trx, a.fecha_trx,
                a.cod_autorizacion, a.motivo_cc, a.voucher, a.referencia, a.fecha_devolucion, a.nro_control_contracargo,
                a.moneda_contracargo, a.mto_contracargo, a.arn_ard, a.fecha_contracargo, a.cod_respuesta_ecommerce,
                a.ind_unicidad_contracargo, a.orden_cc, a.cod_terminal, a.flag_retencion, a.flag_devolucion,
                a.metodo_ingreso, a.correo_contacto, a.razon_social, a.linea_atencion, a.flag_correo,
                b.document_number
            from `{var_project_storage}.{var_dataset_bi_riesgo}.{var_table_dv_consolidacion_contracargo}` a
            left join `{var_project_sensitive}.{var_dataset_master_pii}.{var_table_iden_party_data_control}` b
                 on (a.party_id_izi = b.party_id_izi)
            where a.fecha_contracargo between '{fecha_inicio}' and '{fecha_fin}'
              and a.flag_correo is true
        )
        select
            current_date("America/Lima")                                                                    as process_date,
            cod_comercio                                                                                    as CodComercio,
            case when left(nom_comercio, 13) = 'ECM*AMZ VIDEO' then '************'
                 else concat(bin, '******', cuarteto) end                                                 as Tarjeta_Encriptada,
            nom_comercio                                                                                    as NomComercio,
            moneda_trx                                                                                      as MonedaTrx,
            mto_trx                                                                                         as ImporteTrx,
            fecha_trx                                                                                       as FechaTrx,
            cod_autorizacion                                                                                as CodAutorizacion,
            motivo_cc                                                                                       as MotivoCC,
            voucher                                                                                         as Voucher,
            referencia                                                                                      as Referencia,
            fecha_devolucion                                                                                as FechaDev,
            nullif(upper(trim(AEAD.DECRYPT_STRING(sec_name.key, razon_social, sec_name.constant))),'')      as RazonSocial,
            nro_control_contracargo                                                                         as NroControl_ROLCase,
            case when left(nom_comercio, 13) = 'ECM*AMZ VIDEO' then '************'
                 else concat(bin, '******', cuarteto) end                                                 as Tarjeta,
            moneda_contracargo                                                                              as MonedaCC,
            mto_contracargo                                                                                 as ImporteCC,
            arn_ard                                                                                         as ARD_ARN,
            fecha_contracargo                                                                               as FechaCC,
            cod_respuesta_ecommerce                                                                         as Ecom,
            ind_unicidad_contracargo                                                                        as Contracargo,
            orden_cc                                                                                        as Orden,
            cod_terminal                                                                                    as Terminal,
            case when flag_retencion is true and flag_devolucion is true then 'RET Y DEV'
                 when flag_retencion is true                             then 'RETENCION'
                 when flag_devolucion is true                            then 'DEVOLUCION'
            end                                                                                             as Reten_Dev,
            metodo_ingreso                                                                                  as Entrymode,
            nullif(upper(trim(AEAD.DECRYPT_STRING(sec.key, correo_contacto,sec.constant))),'')              as Contacto_Opera,
            linea_atencion                                                                                  as Tipo_Linea,
            null                                                                                            as Correo_Responsable,
            flag_correo                                                                                     as Flag_correo,
            document_number                                                                                 as RUC,
            'CONTRACARGOS VISA & MASTERCARD'                                                                as record_source,
            current_datetime("America/Lima")                                                                as load_date,
            session_user()                                                                                  as creation_user
        from base
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec_name  on sec_name.code  = 'C_FULL_NAME'
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec on sec.code = 'C_EMAIL';
    """,

    "reporte_cc_cartas_x_comercio": f"""
        with base as (
            select
                a.cod_comercio, a.nom_comercio, a.bin, a.cuarteto, a.moneda_trx, a.mto_trx, a.fecha_trx,
                a.cod_autorizacion, a.motivo_cc, a.voucher, a.referencia, a.fecha_devolucion, a.nro_control_contracargo,
                a.moneda_contracargo, a.mto_contracargo, a.arn_ard, a.fecha_contracargo, a.marca_tarjeta,
                a.categoria_motivo_contracargo, a.cod_respuesta_ecommerce, a.ind_unicidad_contracargo, a.orden_cc,
                a.cod_terminal, a.flag_retencion, a.flag_devolucion, a.metodo_ingreso, a.correo_contacto,
                a.razon_social, a.linea_atencion, a.flag_correo, b.document_number
            from `{var_project_storage}.{var_dataset_bi_riesgo}.{var_table_dv_consolidacion_contracargo}` a
            left join `{var_project_sensitive}.{var_dataset_master_pii}.{var_table_iden_party_data_control}` b
                 on (a.party_id_izi = b.party_id_izi)
            where a.fecha_contracargo >= '{fecha_inicio}'
              and a.fecha_contracargo <= '{fecha_fin}'
              and a.flag_correo is false
              and a.fecha_devolucion is null
              and b.document_number in (
                    '20100070970','20331066703','20394077101','20493020618','20506035121',
                    '20511315922','20512002090','20536557858','20556246743','20600414276',
                    '20601233488','20603150954','20607607061','20608300393','20608430301'
              )
              and not (a.categoria_motivo_contracargo = 'FRAUDE' and a.metodo_ingreso like '%CHIP%')
              and not (a.categoria_motivo_contracargo = 'FRAUDE' and a.metodo_ingreso = 'PQ')
              and a.motivo_cc not in ('12.1 PROCESSING ERROR - LATE PRESENTMENT','PRESENTACIÓN TARDÍA')
           -- and length(AEAD.DECRYPT_STRING(c.key, a.correo_contacto, c.constant)) > 6
           -- and concat(substr(a.marca_tarjeta,1,1), substr(a.categoria_motivo_contracargo,1,1), substr(a.cod_respuesta_ecommerce,1,3)) not in ('MF211','MF212')
           -- and concat(substr(a.marca_tarjeta,1,1), substr(a.categoria_motivo_contracargo,1,1), substr(a.cod_respuesta_ecommerce,1,1)) not in ('VF5','VF6')
        )
        select
            current_date("America/Lima")                                          as process_date,
            cod_comercio                                                          as CodComercio,
            case when left(nom_comercio, 13) = 'ECM*AMZ VIDEO' then '************'
                 else concat(bin, '******', cuarteto) end                       as Tarjeta_Encriptada,
            nom_comercio                                                          as NomComercio,
            moneda_trx                                                            as MonedaTrx,
            mto_trx                                                               as ImporteTrx,
            fecha_trx                                                             as FechaTrx,
            cod_autorizacion                                                      as CodAutorizacion,
            motivo_cc                                                             as MotivoCC,
            voucher                                                               as Voucher,
            referencia                                                            as Referencia,
            fecha_devolucion                                                      as FechaDev,
            AEAD.DECRYPT_STRING(sec_name.key, razon_social,sec_name.constant)     as RazonSocial,
            nro_control_contracargo                                               as NroControl_ROLCase,
            case when left(nom_comercio, 13) = 'ECM*AMZ VIDEO' then '************'
                 else concat(bin, '******', cuarteto) end                       as Tarjeta,
            moneda_contracargo                                                    as MonedaCC,
            mto_contracargo                                                       as ImporteCC,
            arn_ard                                                               as ARD_ARN,
            fecha_contracargo                                                     as FechaCC,
            cod_respuesta_ecommerce                                               as Ecom,
            ind_unicidad_contracargo                                              as Contracargo,
            orden_cc                                                              as Orden,
            cod_terminal                                                          as Terminal,
            case when flag_retencion is true and flag_devolucion is true then 'RET Y DEV'
                 when flag_retencion is true                             then 'RETENCION'
                 when flag_devolucion is true                            then 'DEVOLUCION'
            end                                                                   as Reten_Dev,
            metodo_ingreso                                                        as Entrymode,
            AEAD.DECRYPT_STRING(sec.key, correo_contacto, sec.constant)           as Contacto_Opera,
            linea_atencion                                                        as Tipo_Linea,
            AEAD.DECRYPT_STRING(sec.key, correo_contacto, sec.constant)           as Correo_Responsable,
            flag_correo                                                           as Flag_correo,
            document_number                                                       as RUC,
            'CONTRACARGOS VISA & MASTERCARD'                                      as record_source,
            current_datetime("America/Lima")                                      as load_date,
            session_user()                                                        as creation_user
        from base
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec_name on sec_name.code  = 'C_FULL_NAME'
        left join `{var_project_sensitive}.{var_dataset_secure_secrets}.{var_table_config_protected_data}` sec on sec.code = 'C_EMAIL';
    """
}

--- Año del proceso: 2026 ---
--- Mes del proceso: 04 ---
--- Fecha para nombres de archivos: 20260406 ---


In [ ]:
# =============================================================================
# 4. Ejecución de Exportación y Consolidación (Bucle para todos los reportes)
# =============================================================================
bucket = storage_client.bucket(bucket_name)
print(f"--- Iniciando proceso iterativo para las fechas: {fecha_inicio} a {fecha_fin} ---\n")

for nombre_reporte, query_base in reportes.items():
    nombre_final = f"{nombre_reporte}_{var_fecha_file}.csv"
    uri_temporal = f"gs://{bucket_name}/{ruta_base}temp_{nombre_reporte}_{var_fecha_file}_*.csv"

    print(f"========================================================")
    print(f"🚀 Procesando reporte: {nombre_reporte}")
    print(f"========================================================")

    query_export = f"""
    EXPORT DATA OPTIONS (
      uri = '{uri_temporal}',
      format = 'CSV',
      overwrite = true,
      header = true
    ) AS
    {query_base}
    """

    # 4.1 Exportar desde BigQuery
    print("Paso 1: Exportando particiones desde BigQuery a GCS...")
    clientBQ.query(query_export).result()

    # 4.2 Consolidar archivos en Pandas
    print("Paso 2: Consolidando archivos temporales...")
    prefix_temp = f"{ruta_base}temp_{nombre_reporte}_{var_fecha_file}_"
    blobs = list(bucket.list_blobs(prefix=prefix_temp))

    if not blobs:
        print(f"⚠️ AVISO: No se encontraron archivos temporales para {nombre_reporte}.")
        continue

    lista_df = []
    skipped = 0
    for blob in blobs:
        if blob.size == 0:
            skipped += 1
            continue

        uri_parte = f"gs://{bucket_name}/{blob.name}"
        try:
            # header=0 infiere automáticamente los nombres de columna del csv exportado
            df_parte = pd.read_csv(uri_parte, header=0)

            # 👇 CAMBIO MÍNIMO AQUÍ 👇
            # Agregamos siempre el dataframe para no perder las cabeceras
            lista_df.append(df_parte)

            if df_parte.empty:
                skipped += 1 # Solo lo contamos como informativo
            # 👆 FIN DEL CAMBIO 👆

        except Exception as e:
            print(f"⚠️ Error leyendo {blob.name}: {e}")
            skipped += 1

    if not lista_df:
        print(f"❌ ERROR: Todos los archivos temporales fallaron al leerse para {nombre_reporte}.")
    else:
        print(f"📦 Partes leídas (incluyendo vacías): {len(lista_df)} | Sin datos: {skipped}")

        # Al concatenar, si todos los df están vacíos, Pandas crea un df vacío pero CON las columnas intactas
        df_final = pd.concat(lista_df, ignore_index=True)

        ruta_final_full = f"gs://{bucket_name}/{ruta_base}{nombre_final}"
        # Al guardar el CSV de un df vacío, se imprime solo la cabecera automáticamente
        df_final.to_csv(ruta_final_full, index=False, sep=";", encoding="utf-8-sig")
        print(f"✅ ÉXITO: Archivo creado en: {ruta_final_full}")

        # 4.3 Limpieza: Borrar temporales de este reporte específico
        for blob in blobs:
            try:
                bucket.blob(blob.name).delete()
            except Exception as e:
                print(f"⚠️ No se pudo borrar {blob.name}: {e}")

print("\n🏁 Todos los reportes han sido generados y consolidados exitosamente.")

# =============================================================================
# 5. Limpieza preventiva: Borrar cualquier archivo 'temp' residual en la ruta
# =============================================================================
print(f"🧹 Iniciando barrido de archivos basura (temp_*) en: {ruta_base}")
blobs_basura = bucket.list_blobs(prefix=ruta_base)

contador_borrados = 0
for blob in blobs_basura:
    nombre_archivo = blob.name.split('/')[-1]

    if nombre_archivo.startswith("temp"):
        try:
            blob.delete()
            contador_borrados += 1
        except Exception as e:
            print(f"⚠️ Error al borrar {blob.name}: {e}")

print(f"✨ Barrido completado. Se eliminaron {contador_borrados} archivos temporales antiguos.\n")

--- Iniciando proceso iterativo para las fechas: 2026-04-03 a 2026-04-03 ---

🚀 Procesando reporte: reporte_cc_primera_linea
Paso 1: Exportando particiones desde BigQuery a GCS...


Forbidden: 403 GET https://bigquery.googleapis.com/bigquery/v2/projects/prd-izipay-data-operation/queries/7c89ade2-a6a9-43e7-988f-efce53f5d583?maxResults=0&location=US&prettyPrint=false: Access Denied: BigQuery BigQuery: User has neither fine-grained reader nor masked get permission to get data protected by policy tag "Izi_Politica_Mask : Alta" on columns prd-izipay-data-sensitive.secure_secrets.config_protected_data.constant, prd-izipay-data-sensitive.secure_secrets.config_protected_data.key.

Location: US
Job ID: 7c89ade2-a6a9-43e7-988f-efce53f5d583
